# SD_5min Edge Delay Analysis

这个 notebook 直接在 `SD_5min` 上，按论文中的 `max-cross-correlation (MCC)` 口径分析边传播 delay 分布。

这里采用更贴近论文的实现：先用 **natural cubic spline** 生成连续路径，再在更细时间网格上搜索 delay。


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

REPO_ROOT = Path.cwd()
sys.path.append(str(REPO_ROOT / 'benchmark' / 'eval'))

from old_sd_analysis_utils import (
    DEFAULT_SD_5MIN_DIR,
    DEFAULT_SD_5MIN_GRAPH_ROOT,
    build_sd_5min_graph_matrices,
    compute_delay_distribution,
    load_dataset_flow,
    load_sensor_ids,
    summarize_delay_distribution,
)

sns.set_theme(style='whitegrid')


In [ ]:
# Config
SD_5MIN_DIR = DEFAULT_SD_5MIN_DIR
GRAPH_ROOT = DEFAULT_SD_5MIN_GRAPH_ROOT

# 论文口径：先 spline 到更细粒度，再做 MCC
# 对 5min 数据，默认细化到 1min
INTERP_MINUTES = 1
MAX_LAG_MINUTES = 60


In [ ]:
flow, desc = load_dataset_flow(SD_5MIN_DIR)
sensor_ids = load_sensor_ids(SD_5MIN_DIR, int(desc['num_nodes']))
graphs = build_sd_5min_graph_matrices(SD_5MIN_DIR, GRAPH_ROOT)
native_minutes = int(desc['frequency (minutes)'])
native_minutes


In [ ]:
delay_frames = []
for graph_name in ['distthre', 'phys_dir', 'phys_bidir']:
    delay_frames.append(
        compute_delay_distribution(
            graph_name=graph_name,
            adj=graphs[graph_name],
            flow=flow,
            sensor_ids=sensor_ids,
            native_minutes=native_minutes,
            interp_minutes=INTERP_MINUTES,
            max_lag_minutes=MAX_LAG_MINUTES,
            interpolation_method='natural_cubic_spline',
        )
    )

delay_df = pd.concat(delay_frames, ignore_index=True)
delay_df.head()


In [ ]:
delay_summary = summarize_delay_distribution(delay_df)
delay_summary


In [ ]:
plt.figure(figsize=(12, 5))
sns.histplot(
    data=delay_df,
    x='delay_minutes',
    hue='graph',
    bins=20,
    element='step',
    common_norm=False,
)
plt.title('SD_5min Edge Delay Distribution (MCC-based)')
plt.show()


In [ ]:
plt.figure(figsize=(12, 5))
sns.boxplot(data=delay_df, x='graph', y='delay_minutes')
plt.title('SD_5min Edge Delay by Graph')
plt.show()


In [ ]:
plt.figure(figsize=(12, 5))
sns.histplot(
    data=delay_df,
    x='corr_peak',
    hue='graph',
    bins=20,
    element='step',
    common_norm=False,
)
plt.title('SD_5min Peak Correlation Distribution of Delay Estimation')
plt.show()
